# 01 - Ingest Bronze Layer

This step shows the process for collecting data from **Home Credit Default Risk** on [Kaggle](https://www.kaggle.com/competitions/home-credit-default-risk/data) and ingest the required data in a Bronze Delta Tables.

This notebook verifies access to the Home Credit sources, records the granularity and keys of the three selected files, and persists them. The Bronze layer preserves the source fields without applying cleaning rules; any issues encountered are logged for handling in the Silver layer.

- **Goal:** Make `application_train`, `bureau` and `previous_application` data available in Delta Bronze tables, preserving source fields and recording metadata load provenance.
- **Inputs:** CSV files in the Volume named `landing.home_credit_default_risk`
- **Outputs:** Delta Tables in the `bronze` schema and an initial quality summary. 
- **Step Constraints:** Not convert business attributes, deduplicate histories or convert features.

## Settings
This section includes package imports and global variables useful for the entire ingestion process, such as catalog, schema and volume names. It also include initial checks of the catalog architecture, preserving the proper structure of the medallion architecture.

In [0]:
## Imports
import kagglehub
from pyspark.sql import functions as F

In [0]:
## Setting up the catalog, schemas and volume
CATALOG = "credit_risk_lakehouse"
LANDING_SCHEMA = "landing"
BRONZE_SCHEMA = "bronze"

VOLUME_NAME = "home_credit_default_risk"
VOLUME_PATH = f"/Volumes/{CATALOG}/{LANDING_SCHEMA}/{VOLUME_NAME}"

RUN_DOWNLOAD = True

SOURCES = {
    "application_train": "application_train.csv",
    "bureau": "bureau.csv",
    "previous_application": "previous_application.csv",
}

In [0]:
## Creating the catalog, schemas and volume
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"USE CATALOG {CATALOG}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_SCHEMA}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {LANDING_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {LANDING_SCHEMA}.{VOLUME_NAME}")

In [0]:
%sql
-- Check catalog and current schema
SELECT current_catalog(), current_schema();

In [0]:
## Check bronze schema
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA};")
spark.sql("SHOW TABLES").display()

# spark.catalog.setCurrentDatabase(BRONZE_SCHEMA)
# spark.catalog.listTables(BRONZE_SCHEMA)

In [0]:
##  Check landing volume
spark.sql(f"USE SCHEMA {LANDING_SCHEMA}")
spark.sql(f"SHOW VOLUMES").display()

## Collecting CSV files
In this section, we are connecting to Kaggle Platform through the `kagglehub` API. A Kaggle account and a configured API key are required to download the data. We are downloading the entire available dataset, but initially we will work only with three CSV files, as previously mentioned. As the project evolves, we will ingest the remaining tables.

In [0]:
if RUN_DOWNLOAD:
    # Logging into Kaggle (it may take a few seconds)
    # It requires a Kaggle account and an API key
    kagglehub.login()

In [0]:
if RUN_DOWNLOAD:
    # Downloading the data into the volume
    kagglehub.competition_download(
        "home-credit-default-risk",
        output_dir=VOLUME_PATH,
    )

In [0]:
## Checking the download of the files
spark.sql(f"LIST '{VOLUME_PATH}'").display()

## Data preview inspection
Here, we're verifying the dimensions, headers and few samples of the raw data prior to persistence into the Bronze layer.

All business columns are read without inferring schema (`inferSchema = False`), retaining their data type as strings and their values "as-is" for preserving the original source content.

Decisions regarding data typing and the interpretation of special values will be documented in the Silver Layer. The counts presented in this section will serve to validate the ingestion into the Bronze layer .

In [0]:
## Helper function to preview the content of a file 
def show_file_content(path):
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv(path)
    )

    print(f"Row count: {df.count()} | Number of columns: {len(df.columns)}")
    df.printSchema()
    display(df.limit(10))

    del df

In [0]:
## Inspecting the content of application_train
path = f"{VOLUME_PATH}/application_train.csv"
show_file_content(path)

In [0]:
## Inspecting the content of bureau file
path = f"{VOLUME_PATH}/bureau.csv"
show_file_content(path)

In [0]:
## Inspecting content of previous_application file
path = f"{VOLUME_PATH}/previous_application.csv"
show_file_content(path)

## Bronze snapshot persistence
Each selected CSV file generates a managed Delta Table in the `bronze` schema. The execution replaces completely the snapshot of the data while preserving the table identity across loads, without any cleaning or processing step; the `_rescued_data` allows for the identification of records that did not fit the read schema.

As we can notice, the creation of the Delta tables automatically provides metadata records, such as a table ID, creation date (`createdAt`), last modification date (`lastModified`), table size and other history details. Additionally, we include the original source path (`_source_path`) and ingetion date (`ingested_at`). This is important for traceability, governance and data maintenance.

In [0]:
## Creating delta tables into the bronze schema
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

for delta_table, filename in SOURCES.items():
    print(f"Loading {filename} into {delta_table}")
    
    # 'Create table as' statement
    spark.sql(f"""
        CREATE OR REPLACE TABLE {delta_table}
        USING DELTA AS
        SELECT
        src.*,
        src._metadata.file_path AS _source_path,
        current_timestamp() AS _ingested_at
        FROM read_files(
            '{VOLUME_PATH}/{filename}',
            format => 'csv',
            header => true,
            inferSchema => false
        ) AS src;
    """)

    # Checking the table
    display(spark.sql(f"DESCRIBE DETAIL {delta_table}"))
    display(spark.sql(f"DESCRIBE HISTORY {delta_table}"))


In [0]:
## Display available tables in your schema
spark.sql("SHOW TABLES;").display()

## Quick analysis
In this section, we are performing a quick data check. Since the data was loaded without inferring data type, all columns are treated as strings, which prevents numerical analysis.

Therefore, at this stage, we simply check the overall record count, column count, the number and percentage of null values, and the count of distinct values for each column.

Additionally, we examine the cardinality/relationship between tables as part of a quality check, as well as analyzing the `target` column distribution on `application_train` which will be necessary for ML model selection and training. 

### Count, Unique and Null checks

In [0]:
## Helper function for exploring the bronze layer
def analyze_bronze_layer(table_name: str):
    df = spark.table(table_name)
    total_rows = df.count()
    rescued_rows = df.filter(F.col("_rescued_data").isNotNull()).count()
    rescued_pct = (rescued_rows / total_rows) * 100

    print(f"📊 ANÁLISE DA TABELA: {table_name}")
    print(f"Dimensões: {total_rows} linhas x {len(df.columns)} colunas")
    print(f"Corrompidos: {rescued_rows} ({rescued_pct:.2f}%)")

    # List of expressions
    expressions = []

    for c in df.columns:
        expressions.extend([
            F.count(F.col(c)).alias(f"{c}_total"),
            F.count_distinct(F.col(c)).alias(f"{c}_uniques"),
        ])

    # Execute aggregations in an unique query
    summary = df.select(expressions).collect()[0]

    # Organize the results in a table
    summary_rows = []
    for c in df.columns:
        filled = summary[f"{c}_total"]
        nulls = total_rows - filled
        pct_nulls = (nulls / total_rows) * 100
        uniques = summary[f"{c}_uniques"]

        summary_rows.append((c, filled, nulls, pct_nulls, uniques))

    # Create a DataFrame with the summary
    summary_df = spark.createDataFrame(
        summary_rows,
        schema=["Column", "Total", "Nulls", "% Nulls", "Uniques"]
    )

    return summary_df

In [0]:
for table_name in SOURCES.keys():
    df = analyze_bronze_layer(f"{BRONZE_SCHEMA}.{table_name}")
    display(df)
    del df

### Cardinality check

In [0]:
%sql
-- Checking customer coverage of application_train in the bureau table
SELECT 
    COUNT(DISTINCT app.SK_ID_CURR) as total_customer_app,
    COUNT(DISTINCT bu.SK_ID_CURR) as customer_with_history_bureau,
    (COUNT(DISTINCT bu.SK_ID_CURR) / COUNT(DISTINCT app.SK_ID_CURR)) * 100 as pct_customer_with_history
FROM credit_risk_lakehouse.bronze.application_train app
LEFT JOIN credit_risk_lakehouse.bronze.bureau bu 
    ON app.SK_ID_CURR = bu.SK_ID_CURR;

In [0]:
%sql
-- Checking customer coverage of application_train in the previous_application table
SELECT 
    COUNT(DISTINCT app.SK_ID_CURR) as total_customer_app,
    COUNT(DISTINCT papp.SK_ID_CURR) as customer_with_previous_app,
    (COUNT(DISTINCT papp.SK_ID_CURR) / COUNT(DISTINCT app.SK_ID_CURR)) * 100 as pct_customer_with_previous_app
FROM credit_risk_lakehouse.bronze.application_train app
LEFT JOIN credit_risk_lakehouse.bronze.previous_application papp
    ON app.SK_ID_CURR = papp.SK_ID_CURR;

### Target column distribution

In [0]:
application_df = spark.table(f"{BRONZE_SCHEMA}.application_train")
display(
    application_df
    .groupby(F.col("TARGET"))
    .count()
    .orderBy(F.col("TARGET"))
)
del application_df

## Findings for the next step
- `application_train` table contains 307.511 applications rows, with all records populated with a unique `SK_ID_CURR` identifier, without duplicates.
- The `TARGET` column in `application_train` table shows a consistent distribution of values, with no null or invalid entries;  the distribution shows 282.686 records (around 92%) with the value `0` and 24.825 records (around 8%) with the value `1`.
- In the `application_train` table, a large number of columns show a predominance of null values.
- `bureau` table contains 1.716.428 credits rows, with all records populated with a unique `SK_ID_BUREAU` identifier, without duplicates. History exists exists for 85,69% of applications in the main dataset table.
- `previous_applications` table contains 1.670.214 previous proposals rows with all records populated with a unique `SK_ID_PREV` identifier, without duplicates. History exists for 94,65% of applications in the main dataset table.
- `_rescued_data` column is null in all three tables for this load, this shows that the initial ingestion process occurred successfully.
- Fields with **high rates of missing values** — such as `bureau.AMT_ANNUITY` and `previous_application.RATE_INTEREST_PRIMARY` — require an assessment of their significance before deciding whether to impute or discard them when selecting features for training a ML model.
- In the Silver layer, it is required checking for sentinel values, whitespace, categorical domains, and data type conversions. Historical clients with no match in `application_train` should be examined within the context of the competition's scope.